# Extracting Text with PyMuPDF

**Lesson 5 — Module 1: Extraction**

In this notebook you'll use [PyMuPDF](https://pymupdf.readthedocs.io/) to extract text from email PDFs.

## What You'll Learn

- How to extract embedded text from a PDF with PyMuPDF
- What "clean" and "dirty" extraction output looks like
- How fast plain-text extraction is compared to other methods

## Important

This notebook focuses on **understanding** PyMuPDF on a small number of files. We'll run the full extraction across the entire corpus in a later notebook.

## 1. Install Dependencies

In [ ]:
%pip install pymupdf ipywidgets -q

## 2. Imports & Configuration

In [5]:
import pymupdf
from pathlib import Path

PDF_DIR = Path("enron_pdfs")
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files):,} PDFs in {PDF_DIR.resolve()}")

Found 4,911 PDFs in /Users/henryadamcollie/Documents/GitHub/1_pdf_to_kg_notebooks/enron_pdfs


## 3. Extract a Clean PDF

PyMuPDF reads the **embedded text layer** directly from a PDF — no OCR, no image processing.

Let's start with a digitally generated PDF to see what clean output looks like.

In [6]:
# Pick a small file — these are digitally generated PDFs with clean text layers
clean_pdfs = [p for p in pdf_files if p.stat().st_size < 50_000]
sample_pdf = clean_pdfs[0]

doc = pymupdf.open(sample_pdf)
page_count = len(doc)
text = "\n\n".join(page.get_text() for page in doc)
doc.close()

print(f"File:       {sample_pdf.name}")
print(f"Size:       {sample_pdf.stat().st_size / 1024:.1f} KB")
print(f"Pages:      {page_count}")
print(f"Characters: {len(text):,}")
print("=" * 60)
print(text[:2000])

File:       E0000813E.pdf
Size:       2.4 KB
Pages:      1
Characters: 945
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0000813E
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Rob Bradley <rob.bradley@enron.com>
Sent:
Tue, 21 Nov 2000 08:34:00 -0800 (PST)
To:
Kenneth Lay <kenneth.lay@enron.com>
Cc:
Alhamd Alkhayat <alhamd.alkhayat@enron.com>
Subject:
Remarks to EES Employees--December 1
[REDACTED] B6
EES employee meeting next Friday, and I put together the attached as a
potential framework.  It provides an Enron problem list of 1985/86 versus
today and a brief look at our different visions.
If I can do something else, I'll be back Monday morning and will be happy to
do so.
- Rob
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0000813E
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.

### Try a few more

Run the cell below to look at a handful of clean PDFs. Check for:

- Email header visibility (From, To, Sent, Subject)
- Body text readability
- Obvious errors or garbled characters

In [7]:
for pdf_path in clean_pdfs[:5]:
    doc = pymupdf.open(pdf_path)
    page_count = len(doc)
    text = "\n\n".join(page.get_text() for page in doc)
    doc.close()

    print(f"\n{'=' * 60}")
    print(f"File: {pdf_path.name}  |  Pages: {page_count}  |  Chars: {len(text):,}")
    print("=" * 60)
    print(text[:500])
    print("...")


File: E0000813E.pdf  |  Pages: 1  |  Chars: 945
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0000813E
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Rob Bradley <rob.bradley@enron.com>
Sent:
Tue, 21 Nov 2000 08:34:00 -0800 (PST)
To:
Kenneth Lay <kenneth.lay@enron.com>
Cc:
Alhamd Alkhayat <alhamd.alkhayat@enron.com>
Subject:
Remarks to EES Employees--December 1
[REDACTED] B6
EES employee meeting next Friday, and I put together t
...

File: E0000D1D0.pdf  |  Pages: 2  |  Chars: 2,843
CONFIDENTIAL
Enron Corp.
Case No. EC-2002-01038
Doc No. E0000D1D0
Date: 01/15/2003
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
CONFIDENTIAL - SUBJECT TO PROTECTIVE ORDER
From:
EDIS Email Service <edismail@incident.com>
Sent:
Fri, 5 Apr 2002 00:07:00 -0800 (PST)
To:
Motley, Matt <matt.motley@enron.com>
Subject:
[EDIS]  EQ 4 2

### Now look at a scanned one

Not all PDFs in our dataset were born digital. Some are scanned images that were previously run through OCR software — which embedded a text layer into the PDF.

PyMuPDF doesn't know the difference. It faithfully extracts whatever text layer it finds, whether it came from a digital export or from OCR on a scanned image.

In [ ]:
# Larger files are scanned images with an existing OCR text layer
sample_ocr = PDF_DIR / "E00CF8AE9.pdf"

doc = pymupdf.open(sample_ocr)
page_count = len(doc)
text = "\n\n".join(page.get_text() for page in doc)
doc.close()

print(f"File:       {sample_ocr.name}")
print(f"Size:       {sample_ocr.stat().st_size / 1024:.0f} KB  <- much larger (contains page images)")
print(f"Pages:      {page_count}")
print(f"Characters: {len(text):,}")
print("=" * 60)
print(text[:3000])

File:       E00CF8AE9.pdf
Size:       142 KB  ← much larger (contains page images)
Pages:      2
Characters: 3,205
CONFIDENTIAL 
Enron Corp. 
Case No. EC-2002-01038 
Doc No. EOOCF8AE9 
Date: 01/15/2003 
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA. 
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED. 
RELEASE IN PART 
From: 
Amr Ibrahim <amr.ibrahim@enron.com> 
Sent: 
Thu, 5 Jul 2001 01:02:00 -0700 (PDT) 
To: 
Richard Shapiro <richard.shapiro@enron.com>, Lisa Yoho 
<lisa.yoho@enron.com>, [REDACTED] Bé6 
Subject: 
Fireball Coal Project - RAC Meeting June 29 
Rick: 
This is quick note to let you know that I shall work very closely with Lisa 
on Fireball project (see first project below). 
While working in her group to 
[REDACTED] B5 
measure to reflect the risk the Enron sees for GA reporting purposes (also 
measures for our contribution to the project). 
Not only this project is important in terms of size ($750 million), but also 
in its activity (acquisition of coal comp

### What's different?

Compare the output above with the clean extraction from `E0048ADF3.pdf`. The text is mostly correct — but look closely and you'll spot subtle OCR errors:

| What it should say | What OCR produced | Error type |
|---|---|---|
| `Doc No. E00CF8AE9` | `Doc No. EOOCF8AE9` | `0` → `O` (zero vs letter O) |
| `[REDACTED] B6` | `[REDACTED] Bé6` | Accent artifact |

These are typical OCR errors on clean B&W scans: character-level substitutions where visually similar shapes get confused. At 300 DPI, Tesseract gets most things right — but `0`/`O`, `1`/`l`, `5`/`S` confusions are common.

Not all scans are this clean. Some files in the dataset have **much worse** OCR from lower-quality scans — `Bate:` instead of `Date:`, `ENRON CORE.` instead of `ENRON CORP.`, or headers garbled beyond recognition. We'll see the full range in the next cell.

PyMuPDF did its job perfectly — it extracted exactly what was embedded. The quality depends entirely on how the OCR was originally done.

These may seem like small issues -- worth ignoring. But, at scale, these data-quality issues can cascade, compound, and ultimately make life harder, further down the road. 

### The quality spectrum

OCR quality varies across the dataset. Some files have nearly perfect text with occasional character swaps. Others have garbled headers from lower-quality scans. Here are three examples spanning that range:

In [6]:
# Examples across the OCR quality spectrum
ocr_examples = [
    PDF_DIR / "E00CF8AE9.pdf",   # clean scan — subtle errors (0→O in doc ID)
    PDF_DIR / "E61D04918.pdf",   # moderate — Bate:, ENRON CORE., EROTECTIVE
    PDF_DIR / "ECD0D46C3.pdf",   # severe — COMP DDENTIAL, EMURCN CURD.
]

for pdf_path in ocr_examples:
    doc = pymupdf.open(pdf_path)
    page_count = len(doc)
    pages = [page.get_text() for page in doc]
    text = "\n\n".join(pages)
    doc.close()

    print(f"{'=' * 60}")
    print(f"{pdf_path.name} — {page_count} page(s), {pdf_path.stat().st_size / 1024:.0f} KB")
    print(f"{'=' * 60}")
    print(text[:500])
    print()


E00CF8AE9.pdf — 2 page(s), 142 KB
CONFIDENTIAL 
Enron Corp. 
Case No. EC-2002-01038 
Doc No. EOOCF8AE9 
Date: 01/15/2003 
ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA. 
SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED. 
RELEASE IN PART 
From: 
Amr Ibrahim <amr.ibrahim@enron.com> 
Sent: 
Thu, 5 Jul 2001 01:02:00 -0700 (PDT) 
To: 
Richard Shapiro <richard.shapiro@enron.com>, Lisa Yoho 
<lisa.yoho@enron.com>, [REDACTED] Bé6 
Subject: 
Fireball Coal Project - RAC Meeting June 29 
Rick: 
This is quick note to let you

E61D04918.pdf — 1 page(s), 37 KB
CONFIDENTIAL 
Enron Corp. 
Case No. EC-26002-016038 
Doc No. Es6lbo4918 
Bate: O1/15/2003 
ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA. 
SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED. 
RELEASE IN PART 
From: 
Kerri Thompson <kerri.thompson@enron,. 
com> 
Sent: 
Wed, 22 Nov 2000 05:41:00 -0800 (PST) 
To: 
Kate Symes <kate.symes@enron.com> 
Subject : 
natsource checkout 
465810 
matt 
strike price 70.00 
8.00

Three levels of OCR quality from the same extraction method:

**Clean scan** (`E00CF8AE9`) — nearly perfect. The only visible error is `EOOCF8AE9` in the doc ID, where Tesseract read `0` as `O`. The headers, names, email addresses, and body text are all intact.

**Moderate degradation** (`E61D04918`) — the boilerplate is visibly corrupted (`Bate:` for `Date:`, `ENRON CORE.` for `ENRON CORP.`, `EROTECTIVE` for `PROTECTIVE`), and the email address has been split across two lines with a stray comma: `kerri.thompson@enron,.com>`. The names and body text are still mostly readable.

**Severe degradation** (`ECD0D46C3`) — the text is barely recognisable. `CONFIDENTIAL` has become `COMP DDENTIAL`, `Enron Corp.` is `Engen UlErR.`, and the email address is garbled beyond use: `kiwv warctenron.com`. The date reads `Tnu, 4 Get 2OOL Bfibes:3s`. A human could not reconstruct the original from this text alone.

These three tiers represent the range you'll encounter in most real document dumps. The clean scans are the majority. The moderate cases are where parsing gets interesting. The severe cases are where you'll need a more robust strategy.


### And some with no text at all

Some scanned PDFs were never OCR'd — they're image-only files with no text layer. PyMuPDF returns nothing.

In [7]:
sample_empty = PDF_DIR / "E0033CF3B.pdf"

doc = pymupdf.open(sample_empty)
text = "\n\n".join(page.get_text() for page in doc)
doc.close()

print(f"File:       {sample_empty.name}")
print(f"Size:       {sample_empty.stat().st_size / 1024:.0f} KB")
print(f"Characters: {len(text.strip())}")
print(text[:200] if text.strip() else "(empty — no text layer)")

File:       E0033CF3B.pdf
Size:       1031 KB
Characters: 0
(empty — no text layer)


### Classifying PDFs by structure

We've now seen three kinds of PDF. How do we tell them apart programmatically?

Two signals are enough:
- **File size** — digital PDFs are a few KB (text only). Scanned PDFs are tens to hundreds of KB (they contain page images).
- **Embedded images** — scanned PDFs have at least one image per page. Digital PDFs have none.

In [8]:
def classify_pdf(pdf_path):
    """Classify a PDF by its structure.

    Returns one of:
      'digital'    — no embedded images, text generated directly
      'scanned'    — has page images + OCR text layer
      'image_only' — has page images but no text layer
    """
    doc = pymupdf.open(pdf_path)
    page = doc[0]
    has_text = bool(page.get_text().strip())
    has_images = len(page.get_images()) > 0
    doc.close()

    if not has_images:
        return "digital"
    elif has_text:
        return "scanned"
    else:
        return "image_only"


# Classify every PDF
counts = {"digital": 0, "scanned": 0, "image_only": 0}
for p in pdf_files:
    counts[classify_pdf(p)] += 1

total = len(pdf_files)
print(f"Total PDFs:                  {total:,}")
print(f"Digital (text only):         {counts['digital']:,}  ({counts['digital']/total:.0%})")
print(f"Scanned (image + OCR text):  {counts['scanned']:,}  ({counts['scanned']/total:.0%})")
print(f"Image-only (no text layer):  {counts['image_only']:,}  ({counts['image_only']/total:.0%})")

Total PDFs:                  4,911
Digital (text only):         2,161  (44%)
Scanned (image + OCR text):  1,955  (40%)
Image-only (no text layer):  795  (16%)


**Digital** files have no images — PyMuPDF extracts clean text directly. **Scanned** files have page images plus an OCR text layer that ranges from nearly perfect to heavily garbled. **Image-only** files have page images but no text layer — PyMuPDF returns nothing, and these need OCR.

For the pipeline:
- **Digital + scanned** -> `get_text()` works (fast)
- **Image-only** -> needs OCR (slow)

## When to OCR
If you are working with a dataset composed of scanned documents that received their OCR treatment many moons ago, you may want to switch this to:

- **Digital** -> `get_text()`
- **Any image** -> use OCR

Some older attempts at OCR on poor scans may be so garbled as to be unusable. In such cases, you can just re-run OCR on any file that contains an image, regardless of its text. 

That said, if the OCR errors are consistent, repeating patterns, and the text is still generally readable, you can likely clean most of it during the cleaning and parsing phases later.

This is especially important to bear in mind where you have a corpus of millions of documents. Running OCR on every image could waste time, where a simple parser would do the entire corpus in hours, not weeks.

## 4. Speed Test

In [9]:
import time

BATCH_SIZE = 5000
batch = pdf_files[:BATCH_SIZE]

start = time.perf_counter()

total_chars = 0
total_pages = 0

for pdf_path in batch:
    doc = pymupdf.open(pdf_path)
    total_pages += len(doc)
    text = "\n\n".join(page.get_text() for page in doc)
    total_chars += len(text)
    doc.close()

elapsed = time.perf_counter() - start

print(f"Extracted {BATCH_SIZE} PDFs in {elapsed:.2f} seconds")
print(f"  {BATCH_SIZE / elapsed:.0f} PDFs/sec")
print(f"  {total_pages / elapsed:.0f} pages/sec")
print(f"  {total_chars:,} total characters")

Extracted 5000 PDFs in 12.34 seconds
  405 PDFs/sec
  593 pages/sec
  9,214,184 total characters


### Putting that in perspective

At this rate, you could process:

| Dataset size | Estimated time |
|---|---|
| 5,000 PDFs (our dataset) | ~10–20 seconds |
| 100,000 PDFs | ~5–7 minutes |
| 1,000,000 PDFs | ~1-2 hours |

Compare that to OCR (~5–10 PDFs/sec) or vision LLMs (<1 page/sec). This is why we use PyMuPDF as the **first tier** of our extraction strategy.

## 5. Reusable Extraction Function

Let's wrap the extraction logic into a function we can reuse later when we run the full pipeline.

In [10]:
def extract_text_pymupdf(pdf_path):
    """Extract embedded text from a PDF using PyMuPDF.

    Returns:
        tuple: (text, page_count) where text is the full extracted text
               and page_count is the number of pages in the PDF.
    """
    doc = pymupdf.open(pdf_path)
    text = "\n\n".join(page.get_text() for page in doc)
    page_count = len(doc)
    doc.close()
    return text, page_count

In [11]:
# Quick test
text, pages = extract_text_pymupdf(pdf_files[0])
print(f"{pdf_files[0].name}: {pages} page(s), {len(text):,} chars")

E0000813E.pdf: 1 page(s), 945 chars


## Summary

- **PyMuPDF reads embedded text layers** — fast, free, and dependency-free
- **Digital PDFs** (44%) have clean text — `get_text()` just works
- **Scanned PDFs with OCR** (40%) have text layers with varying quality — from nearly perfect to severely garbled, depending on the original scan quality
- **Image-only PDFs** (16%) have no text layer — `get_text()` returns nothing
- **Structural classification** (has images? has text?) is enough to route documents — no quality score needed at this stage
- **Speed is exceptional** — hundreds of PDFs per second on commodity hardware

In the next lesson, we'll use OCR to recover text from image-only PDFs, and explore what happens when we re-OCR the scanned files.